# サーバレスアプリケーション

AWS Lambdaなどのアプリケーション開発への応用について。

## Lambdalith

複数のAPIエンドポイントのコンピューティングを1つのLambda Functionに集約するパターンをLambdalithという（Lambda + monolith）。  
エンドポイントごとにLambdaを管理するのは負荷が高いため、Webフレームワークを使ったりしてLambdalithにすると楽に開発できる。

かつては公式にアンチパターンと呼ばれていたが、最近は評価が変わってきている。

[サーバーレスマイクロサービスを構築するための設計アプローチの比較 | Amazon Web Services ブログ](https://aws.amazon.com/jp/blogs/news/comparing-design-approaches-for-building-serverless-microservices/)





### Powertools for AWS Lambda

Lambdalithなアプリを作ったり、構造化ログ出力が簡単に実現できたりなど豊富な機能を持つライブラリ

[Powertools for AWS Lambda - AWS Lambda](https://docs.aws.amazon.com/ja_jp/lambda/latest/dg/powertools-for-lambda.html)

:::{dropdown} コード例

```python
from typing import List
import requests
from pydantic import BaseModel, Field
from aws_lambda_powertools import Logger, Tracer
from aws_lambda_powertools.event_handler import APIGatewayRestResolver
from aws_lambda_powertools.utilities.typing import LambdaContext

tracer = Tracer()
logger = Logger()
app = APIGatewayRestResolver(enable_validation=True)
app.enable_swagger(path="/swagger")  # OpenAPI仕様書 

class Todo(BaseModel):  # Pydanticによる型定義とバリデーション
    userId: int
    id_: int = Field(alias="id")
    title: str
    completed: bool


@app.post("/todos") # POST /todos へのルーティング
def create_todo(todo: Todo) -> str:
    response = requests.post("https://jsonplaceholder.typicode.com/todos", json=todo.dict(by_alias=True))
    response.raise_for_status()
    return response.json()["id"]


@app.get("/todos") # GET /todos へのルーティング
def get_todos() -> List[Todo]:
    todo = requests.get("https://jsonplaceholder.typicode.com/todos")
    todo.raise_for_status()
    return todo.json()


def lambda_handler(event: dict, context: LambdaContext) -> dict:
    return app.resolve(event, context)
```

https://docs.aws.amazon.com/powertools/python/latest/core/event_handler/openapi/#swagger-ui をもとに作成

:::

### aws-lambda-web-adapter

aws-lambda-web-adapterを挟むことでLambda独自のイベント形式が変換され、標準的なWeb App Frameworkが使えるようになる。

https://github.com/aws/aws-lambda-web-adapter


例：

- [FastAPI](https://github.com/aws/aws-lambda-web-adapter/tree/main/examples/fastapi)
- [Remix](https://github.com/aws/aws-lambda-web-adapter/tree/main/examples/remix)



Webフレームワークを使うことで、規模が小さいうちはLambdaを使い、規模が大きくなったらWeb Adapterの1行を消せばECSに移行できる  
[スモールスタートで始めるためのLambda×モノリス（Lambdalith） - Speaker Deck](https://speakerdeck.com/akihisaikeda/sumorusutatodeshi-merutamenolambdaxmonorisu)

### Lamby

Lambda Web AdapterではRailsの例はないが、同様のことをやってくれるのがLamby

https://lamby.cloud/docs/quick-start



### Hono

HonoはCloudflare WorkersのようなFaaSを念頭に作られた。Lambdaでも高速に起動する

[AWS Lambda - Hono](https://hono.dev/docs/getting-started/aws-lambda)
